In [ ]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import com.microsoft.spark.fabric
import datetime,pytz

StatementMeta(, 15d43d2b-ccea-4723-bb70-620b65ed40ac, 3, Finished, Available, Finished)

In [ ]:
spark.read.synapsesql("fwh_gold.CONFIGURATION.PROCESSED_AUDIT_LOGS").createOrReplaceTempView("auditTableView")
spark.read.synapsesql("fwh_gold.CONFIGURATION.PROCESSED_ERROR_LOGS").createOrReplaceTempView("errorTableView")
spark.read.synapsesql("fwh_gold.CONFIGURATION.FIELD_MAPPING").createOrReplaceTempView("fieldMappingView")
# spark.read.synapsesql("fwh_gold.CONFIGURATION.TRUX_SILVER_OBJECT_CONFIGURATIONS_V2").createOrReplaceTempView("truxObjectConfigurationView")
errorSchema = spark.table("errorTableView").schema
auditSchema = spark.table("auditTableView").schema

StatementMeta(, 15d43d2b-ccea-4723-bb70-620b65ed40ac, 4, Finished, Available, Finished)

In [ ]:
def getUTCDatetime():
    UtcNow = datetime.datetime.now(pytz.utc)
    return UtcNow.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

In [ ]:
def createAliasQuery(destinationTableName,sourceName):
    aliasColumnList = spark.sql(f"""SELECT CONCAT(Source_Column_Names, ' AS ', Destination_Column_Names)
                                    FROM fieldMappingView WHERE lower(DESTINATION_TABLE) = lower('{destinationTableName}') AND lower(source_Name) = lower('{sourceName}') """).collect()
    aliasColumns = ", ".join([row[0] for row in aliasColumnList])
    return aliasColumns

In [ ]:
def getColumnsInSqlFormat(columns, sourceAlias="source", targetAlias="target"):
    delimiter = ","
    merge_column_statement = ""
    src_columns = ""
    if columns and columns != "":
        src_columns = ", ".join(f"{sourceAlias}.{col}" for col in columns) 
        for index, col in enumerate(columns):
            if index == 0:
                merge_column_statement = f"{targetAlias}.{col} = {sourceAlias}.{col}"
            else:
                merge_column_statement += f", {targetAlias}.{col} = {sourceAlias}.{col}"
    return src_columns, merge_column_statement

In [ ]:
def getColumnsFromMapping(sourceName, destinationTableName, skipCols=None):
    if skipCols is None:
        skipCols = []

    rows = spark.sql(f"""
        SELECT Destination_Column_Names, Source_Column_Names
        FROM fieldMappingView
        WHERE lower(Source_Name) = lower('{sourceName}')
          AND lower(Destination_Table) = lower('{destinationTableName}')
        ORDER BY ID, Destination_Column_Names, Source_Column_Names
    """).collect()

    if not rows:
        raise ValueError(f"No mappings found for {sourceName} -> {destinationTableName}")

    src_cols = [
        r['Source_Column_Names']
        for r in rows if r['Destination_Column_Names'] not in skipCols
    ]

    tgt_cols = [
        r['Destination_Column_Names']
        for r in rows if r['Destination_Column_Names'] not in skipCols
    ]

    targetColumn = ", ".join(tgt_cols + ["Added_On", "Added_By", "Modified_On", "Modified_By"])


    sourceColumn = ", ".join(src_cols + ["Added_On", "Added_By", "Added_On", "Added_By"])

    return sourceColumn, targetColumn


In [ ]:
def getColumnsForMerge(sourceName, destinationTableName, sourceAlias="s", targetAlias="t", skipCols=None):
    if skipCols is None:
        skipCols = ["Added_On", "Added_By", "Modified_On", "Modified_By"]

    rows = spark.sql(f"""
        SELECT Destination_Column_Names, Source_Column_Names
        FROM fieldMappingView
        WHERE lower(Source_Name) = lower('{sourceName}')
          AND lower(Destination_Table) = lower('{destinationTableName}')
        ORDER BY ID, Destination_Column_Names, Source_Column_Names
    """).collect()

    if not rows:
        raise ValueError(f"No mappings found for {sourceName} -> {destinationTableName}")

    src_columns = []
    tgt_columns = []
    merge_pairs = []

    for r in rows:
        dest_col = r["Destination_Column_Names"]
        src_col  = r["Source_Column_Names"]

        if dest_col in skipCols:
            continue

        src_columns.append(f"{sourceAlias}.{src_col}")
        tgt_columns.append(dest_col)
        merge_pairs.append(f"{targetAlias}.{dest_col} = {sourceAlias}.{src_col}")

    
    audit_fields = ["Added_On", "Added_By", "Modified_On", "Modified_By"]
    target_columns_full = tgt_columns + audit_fields

    merge_pairs.append(f"{targetAlias}.Added_On = {sourceAlias}.Added_On")
    merge_pairs.append(f"{targetAlias}.Added_By = {sourceAlias}.Added_By")

    sourceColumn = ", ".join(src_columns)
    targetColumn = ", ".join(target_columns_full)
    mergeColumnStatement = ", ".join(merge_pairs)

    return sourceColumn, targetColumn, mergeColumnStatement

In [ ]:
def loadDataIntoSalesforceDeltaTable(sourceName, objectName, dfSrc, destinationTableName, operationType, keyName,currentAuditMaxId):
    try:        
        # Create a temp view of the source dataframe
        dfSrc.createOrReplaceTempView("tempView")
        sourceName = 'salesforce'
        targetColumns = ", ".join(i for i in dfSrc.columns)
        

        # Initialize variables
        rowsInserted = 0
        rowsUpdated = 0

        if  operationType.lower() == "append":
            print(f"Append Operation Started on {destinationTableName}")
            insert_query = spark.sql(f"INSERT INTO {destinationTableName}({targetColumns}) select {targetColumns} from tempView")
            df_history = spark.sql(f"describe history {destinationTableName} limit 1").first()
            rowsInserted = df_history["operationMetrics"]["numOutputRows"]
            print("Number of Rows Appended :", rowsInserted)            


        elif operationType.lower() == "overwrite":
            print(f"Overwriting Operation Started on {destinationTableName}")
            overwrite_query = spark.sql(f"INSERT OVERWRITE {destinationTableName}({targetColumns}) select {targetColumns} from tempView")
            df_history = spark.sql(f"describe history {destinationTableName} limit 1").first()
            rowsInserted = df_history["operationMetrics"]["numOutputRows"]
            print("Number of Rows Inserted :", rowsInserted)


        elif operationType.lower() == "upsert":
            print(f"Data Load Operation Started on {destinationTableName}")
            
            destinationLakehouseName = destinationTableName.split(".")[0]
            destinationTableName = destinationTableName.split(".")[-1]
            sourceColumn, targetColumn = getColumnsFromMapping(sourceName, destinationTableName, skipCols=None)
            

            dfSrc = removeDuplicateFromSrc(dfSrc,sourceName,destinationTableName,keyName)
            
            dfSrc.createOrReplaceTempView("tempView")
            
            dfSrc = dfSrc.drop("Added_By", "Added_On", "Modified_By", "Modified_On")
            
            
            srcColumns,targetColumns,mergeColumnStatement = getColumnsForMerge(sourceName, destinationTableName, sourceAlias="s", targetAlias="t", skipCols=None)
            keyNames = "','".join(keyName.split(',')).lower()
            joinCondition = ""
            pkColumnList = spark.sql(f"""SELECT Destination_Column_Names FROM fieldMappingView WHERE lower(Source_Name) = lower('{sourceName}') AND lower(Destination_Table) = lower('{destinationTableName}') AND lower(Source_Column_Names) IN ('{keyNames}')""").collect()
            if not pkColumnList:
                joinCondition = " AND ".join([f"t.{col} = s.{col}" for col in keyName.split(',')])
            else:
                joinCondition = " AND ".join([f"t.{col[0]} = s.{col[0]}" for col in pkColumnList])
            
            if isFullLoad == 1:
                print(f"FULL LOAD STARTED FOR {objectName}")
                delete_query = spark.sql(f"DELETE FROM {destinationLakehouseName}.{destinationTableName}")
                insert_query = spark.sql(f"INSERT INTO {destinationLakehouseName}.{destinationTableName}({targetColumn}) select {sourceColumn} from tempView")
                df_history = spark.sql(f"describe history {destinationLakehouseName}.{destinationTableName} limit 1").first()
                rowsInserted = df_history["operationMetrics"]["numOutputRows"]
                print("Number of Rows Inserted :", rowsInserted)
                endTime = getUTCDatetime()

                tempDF = createLogsEntry(sourceName, objectName, destinationLakehouseName, 
                    'BRONZE TABLE', rowsInserted, rowsUpdated, masterPipelineRunID, startTime,endTime,currentAuditMaxId)
            
                return True, '', tempDF
            else:
            
                print(f"Upsert Operation Started on {destinationTableName}")
                mergeQuery = f"""
                MERGE INTO {destinationLakehouseName}.{destinationTableName} t
                USING tempView s ON {joinCondition}
                WHEN MATCHED THEN UPDATE
                SET {mergeColumnStatement},Modified_On = CURRENT_TIMESTAMP, Modified_By = '{masterPipelineRunID}'
                WHEN NOT MATCHED THEN INSERT ({targetColumns}) VALUES ({srcColumns},CURRENT_TIMESTAMP,'{masterPipelineRunID}', CURRENT_TIMESTAMP, '{masterPipelineRunID}')
                """
                print("Executing merge query")
                spark.sql(mergeQuery)
                print(mergeQuery)

        
                df_history = spark.sql(f"DESCRIBE HISTORY {destinationLakehouseName}.{destinationTableName} LIMIT 1").first()
                if df_history:
                    rowsInserted = df_history["operationMetrics"]["numTargetRowsInserted"]
                    rowsUpdated = df_history["operationMetrics"]["numTargetRowsUpdated"]
                    print(f"RowsInserted: {rowsInserted}, RowsUpdated: {rowsUpdated}")

            endTime = getUTCDatetime()

            tempDF = createLogsEntry(sourceName, objectName, destinationLakehouseName, 
                    'BRONZE TABLE', rowsInserted, rowsUpdated, masterPipelineRunID, startTime,endTime,currentAuditMaxId)
            
            return True, '', tempDF

    except Exception as e:
        print(f"Error during data load: {str(e)}")

        # Log error audit data
        logErrorData(sourceName, masterPipelineRunID,
                     str(e),'LOAD_ERROR', getUTCDatetime())
        
        return False, str(e), None

In [ ]:
def loadDataIntoTruxDeltaTable(sourceName, objectName, dfSrc, destinationLakehouseName, destinationTableName, operationType, keyName,currentAuditMaxId,SourceDatabaseName=None):
    try:        
        # Create a temp view of the source dataframe
        # dfSrc=dfSrc.withColumn("SOURCE",lit(SourceDatabaseName))
        dfSrc.createOrReplaceTempView("tempView")
        targetColumns = ", ".join(i for i in dfSrc.columns)

        # Initialize variables
        rowsInserted = 0
        rowsUpdated = 0

        if  operationType.lower() == "append":
            print(f"Append Operation Started on {destinationLakehouseName}.{destinationTableName}")
            insert_query = spark.sql(f"INSERT INTO {destinationLakehouseName}.{destinationTableName}({targetColumns}) select {targetColumns} from tempView")
            df_history = spark.sql(f"describe history {destinationLakehouseName}.{destinationTableName} limit 1").first()
            rowsInserted = df_history["operationMetrics"]["numOutputRows"]
            print("Number of Rows Appended :", rowsInserted)            


        elif operationType.lower() == "overwrite":
            if sourceName.lower() == 'trux':
                print(f"Overwriting Operation Started on {destinationLakehouseName}.{destinationTableName}")
                silverDF = spark.sql(f"SELECT * FROM {destinationLakehouseName}.{destinationTableName} WHERE SOURCE='{SourceDatabaseName}'")
                if silverDF.count() != 0:
                    delete_query = spark.sql(f"DELETE FROM {destinationLakehouseName}.{destinationTableName} WHERE SOURCE='{SourceDatabaseName}'")
                insert_query = spark.sql(f"INSERT INTO {destinationLakehouseName}.{destinationTableName}({targetColumns}) select {targetColumns} from tempView")
                df_history = spark.sql(f"describe history {destinationLakehouseName}.{destinationTableName} limit 1").first()
                rowsInserted = df_history["operationMetrics"]["numOutputRows"]
                print("Number of Rows Inserted :", rowsInserted)
            else:
                print(f"Overwriting Operation Started on {destinationLakehouseName}.{destinationTableName}")
                overwrite_query = spark.sql(f"INSERT OVERWRITE {destinationLakehouseName}.{destinationTableName}({targetColumns}) select {targetColumns} from tempView")
                df_history = spark.sql(f"describe history {destinationLakehouseName}.{destinationTableName} limit 1").first()
                rowsInserted = df_history["operationMetrics"]["numOutputRows"]
                print("Number of Rows Inserted :", rowsInserted)


        elif operationType.lower() == "upsert":

            #dfSrc=dfSrc.withColumn("SOURCE",lit(SourceDatabaseName))
            targetColumns = ", ".join(i for i in dfSrc.columns)
            print(f"Upsert Operation Started on {destinationLakehouseName}.{destinationTableName}")
            
            dfSrc = removeDuplicateFromSrc(dfSrc,sourceName,destinationTableName,keyName)
            
            
            # Drop unnecessary columns from the source dataframe
            dfSrc = dfSrc.drop("ADDED_BY", "ADDED_ON", "MODIFIED_BY", "MODIFIED_ON")
            
            # Get column names and build the merge condition
            srcColumns, mergeColumnStatement = getColumnsInSqlFormat(dfSrc.columns)
            keyNames = "','".join(keyName.split(',')).lower()
            joinCondition = ""
            pkColumnList = spark.sql(f"""SELECT Destination_Column_Names FROM fieldMappingView WHERE lower(Source_Name) = lower('{sourceName}') AND lower(Destination_Table) = lower('{destinationTableName}') AND lower(Source_Column_Names) IN ('{keyNames}')""").collect()
            if not pkColumnList:
                joinCondition = " AND ".join([f"target.{col} = source.{col}" for col in keyName.split(',')])
            else:
                joinCondition = " AND ".join([f"target.{col[0]} = source.{col[0]}" for col in pkColumnList])
            
            joinCondition=joinCondition+" AND target.SOURCE=source.SOURCE "
            #print("joincondition",joinCondition)
            #print("target columns ",targetColumns)
            #print("source columns ",srcColumns)
            
            dfSrc.createOrReplaceTempView("tempView")
            # Construct the merge query
            mergeQuery = f"""
            MERGE INTO {destinationLakehouseName}.{destinationTableName} target 
            USING tempView source ON {joinCondition}
            WHEN MATCHED THEN UPDATE
            SET {mergeColumnStatement},MODIFIED_ON = CURRENT_TIMESTAMP, MODIFIED_BY = '{masterPipelineRunID}'
            WHEN NOT MATCHED THEN INSERT ({targetColumns}) VALUES ({srcColumns},CURRENT_TIMESTAMP,'{masterPipelineRunID}', CURRENT_TIMESTAMP, '{masterPipelineRunID}')
            """

            # mergeQuery = f"""
            # MERGE INTO {destinationLakehouseName}.{destinationTableName} target 
            # USING tempView source ON {joinCondition}
            # WHEN MATCHED THEN UPDATE
            # SET {mergeColumnStatement},target.MODIFIED_ON = CURRENT_TIMESTAMP, target.MODIFIED_BY = '{masterPipelineRunID}'
            # WHEN NOT MATCHED THEN INSERT ({targetColumns}) VALUES ({srcColumns},CURRENT_TIMESTAMP,'{masterPipelineRunID}', CURRENT_TIMESTAMP, '{masterPipelineRunID}')
            # """
            #print(mergeQuery)
            print("Executing merge query")
            spark.sql(mergeQuery)
            print(mergeQuery)

        # Get the number of inserted and updated rows
            df_history = spark.sql(f"DESCRIBE HISTORY {destinationLakehouseName}.{destinationTableName} LIMIT 1").first()
            if df_history:
                rowsInserted = df_history["operationMetrics"]["numTargetRowsInserted"]
                rowsUpdated = df_history["operationMetrics"]["numTargetRowsUpdated"]
                print(f"RowsInserted: {rowsInserted}, RowsUpdated: {rowsUpdated}")

        endTime = getUTCDatetime()
        tempDF = createLogsEntry(sourceName, objectName, destinationLakehouseName, 
                 targetLayer, 0, 0, masterPipelineRunID, startTime,endTime,currentAuditMaxId)
        return True, '', tempDF

    except Exception as e:
        print(f"Error during data load: {str(e)}")

        # Log error audit data
        logErrorData(sourceName, masterPipelineRunID,
                     str(e),'LOAD_ERROR', getUTCDatetime())
        
        return False, str(e), None


In [ ]:
def logErrorData(sourceName, masterPipelineRunID,errorDescription, errorCode, errorLoggedTime):

    try:

        current_max_id = spark.sql("SELECT MAX(ID) AS max_id FROM errorTableView").collect()[0]["max_id"]
        nextMaxID = 1 if current_max_id is None else current_max_id + 1
        endTime = to_timestamp(lit(getUTCDatetime()), 'yyyy-MM-dd HH:mm:ss.SSS')
        
        error_entry ={
            'ID': nextMaxID,
            'PIPELINE_NAME': pipelineName,
            'PIPELINE_RUNID': masterPipelineRunID,
            'SOURCE': sourceName.upper(),
            'DESTINATION': destination.upper(),
            'ERROR_DESCRIPTION': errorDescription,
            'ERROR_CODE': errorCode,
            'ERROR_LOGGED_TIME': errorLoggedTime,
            'ADDED_BY': masterPipelineRunID
        }
        errorDF = spark.createDataFrame([error_entry], errorSchema) \
                       .withColumn("ADDED_ON",lit(endTime))
        
        errorDF.write.mode("append").synapsesql("fwh_gold.CONFIGURATION.PROCESSED_ERROR_LOGS")

        print("Log auditing into SQL (error) is completed")

    except Exception as e:
        print(f"Error while logging error data: {str(e)}")

In [ ]:
def createLogsEntry(sourceName, objectName, destinationLakehouseName, 
                 layer, rowsInserted, rowsUpdated, masterPipelineRunID, startTime,endTime,currentAuditMaxId):

    totalRecords = int(rowsInserted) + int(rowsUpdated)

    startTime = to_timestamp(lit(startTime), 'yyyy-MM-dd HH:mm:ss.SSS')
    endTime = to_timestamp(lit(endTime), 'yyyy-MM-dd HH:mm:ss.SSS')

    audit_entry = {
        'ID': currentAuditMaxId,
        'SOURCE': sourceName.upper(),
        'OBJECT_NAME': objectName,
        'DESTINATION': destinationLakehouseName.upper(),
        'TARGET_LAYER': layer,
        'INSERT_RECORDS_COUNT': int(rowsInserted),
        'UPDATE_RECORDS_COUNT': int(rowsUpdated),
        'TOTAL_RECORDS_COUNT': totalRecords,
        'PIPELINE_NAME': pipelineName,
        'PIPELINE_RUNID': masterPipelineRunID,
        'MODIFIED_BY': masterPipelineRunID
    }
    df = spark.createDataFrame([audit_entry],auditSchema) \
              .withColumn("RUN_START_TIME",lit(startTime)) \
              .withColumn("RUN_END_TIME", lit(endTime)) \
              .withColumn("MODIFIED_ON", lit(endTime))
    return df

In [ ]:
def addAuditColumns(df, pipelineRunID, targetLayer):
    if targetLayer.lower() == 'bronze':
        df = df.withColumn("ADDED_BY",lit(pipelineRunID)) \
            .withColumn("ADDED_ON",current_timestamp())\
            .withColumn("MODIFIED_ON",current_timestamp())\
            .withColumn("MODIFIED_BY",lit(pipelineRunID))
    
    elif targetLayer.lower() == 'silver' or targetLayer.lower() == 'edw':
        df = df.withColumn("ADDED_ON",current_timestamp()) \
            .withColumn("ADDED_BY",lit(pipelineRunID))\
            .withColumn("MODIFIED_ON",current_timestamp()) \
            .withColumn("MODIFIED_BY",lit(pipelineRunID)) 
    
    return df

In [ ]:
def logsAuditData(sourceName, objectName, destinationLakehouseName,layer, rowsInserted, rowsUpdated, pipelineRunId, pipelineName,startTime,endTime,currentAuditMaxId):
    try:
        startTime = to_timestamp(lit(startTime), 'yyyy-MM-dd HH:mm:ss.SSS')
        endTime = to_timestamp(lit(endTime), 'yyyy-MM-dd HH:mm:ss.SSS')

        totalCount = int(rowsInserted) + int(rowsUpdated)
        auditEntry = {
            'ID': currentAuditMaxId,
            'SOURCE': sourceName.upper(),
            'OBJECT_NAME': objectName,
            'DESTINATION': destinationLakehouseName.upper(),
            'TARGET_layer': layer,
            'INSERT_RECORDS_COUNT': int(rowsInserted),
            'UPDATE_RECORDS_COUNT': int(rowsUpdated),
            'TOTAL_RECORDS_COUNT': int(totalCount),
            'PIPELINE_NAME': pipelineName,
            'PIPELINE_RUNID': pipelineRunId,
            'MODIFIED_BY': pipelineRunId
        }
        auditDF = spark.createDataFrame([auditEntry],auditSchema) \
              .withColumn("RUN_START_TIME",lit(startTime)) \
              .withColumn("RUN_END_TIME", lit(endTime)) \
              .withColumn("MODIFIED_ON", lit(endTime))
        
        auditDF.write.mode("append").synapsesql("fwh_gold.CONFIGURATION.PROCESSED_AUDIT_LOGS")
        print("Log auditing into SQL (audit) is completed")
    except Exception as e:
        print(f"Error while logging audit data: {str(e)}")
    

In [ ]:
def createDestinationPKColumns(destinationTableName,sourceName,keyName):
    keyNames = keyName.split(',')
    keyNameCase = [f"'{name.lower()}'" for name in keyNames]
    keyNameList = ', '.join(keyNameCase)
    columnList = spark.sql(f"""SELECT TRIM(CONCAT(Destination_Column_Names))
                                    FROM fieldMappingView WHERE lower(DESTINATION_TABLE) = lower('{destinationTableName}') AND lower(source_Name) = lower('{sourceName}') AND  lower(Source_Column_Names) IN ({keyNameList}) """).collect()
    columns = ",".join([row[0] for row in columnList])
    return columns.split(',')

In [ ]:
def removeDuplicateFromSrc(dfSrc,sourceName,destinationTableName,keyName):
    dfSrc.createOrReplaceTempView('dfSrcTempView')
    pkColumnList = createDestinationPKColumns(destinationTableName,sourceName,keyName)
    windowSpec = Window.partitionBy(*pkColumnList).orderBy(*pkColumnList)
    duplicateDf = dfSrc.withColumn("rnum", row_number().over(windowSpec))
    finalDf = duplicateDf.filter(col("rnum") == 1).drop("rnum")

    return finalDf

In [ ]:
def getSilverColumnName(sourceName,objectName,sourceColumnName):
    query = f"""SELECT DESTINATION_COLUMN_NAMES FROM fieldMappingView
                    WHERE lower(SOURCE_NAME) = lower('{sourceName}') AND lower(OBJECT_NAME) = lower('{objectName}') AND lower(SOURCE_COLUMN_NAMES) = lower('{sourceColumnName}')"""

    destinationColumnDF = spark.sql(query)
    silverColumn = destinationColumnDF.collect()[0]["DESTINATION_COLUMN_NAMES"]
    return silverColumn

In [ ]:
def loadDataToWarehouse(df, operationType, warehouseTablePath):
    if operationType == 'append':
        df.write.mode("append").synapsesql(f"{warehouseTablePath}")
    elif operationType == 'overwrite':
        df.write.mode("overwrite").synapsesql(f"{warehouseTablePath}")

In [ ]:
def loadDataIntoBronzeDeltaTable(sourceName, objectName, dfSrc, destinationLakehouseName, destinationTableName, operationType, keyName,currentAuditMaxId,columnName,SourceDatabaseName=None):
    try:        
        # Create a temp view of the source dataframe
        dfSrc.createOrReplaceTempView("tempView")
        # Initialize variables
        rowsInserted = 0
        rowsUpdated = 0

        if  operationType.lower() == "append":
            print(f"Append Operation Started on {destinationLakehouseName}.{destinationTableName}")
            select_query = spark.sql(f"select {columnName} from tempView")
            if select_query !=0:
                insert_query = spark.sql(f"INSERT INTO {destinationLakehouseName}.{destinationTableName}({columnName},ISDELETED,MODIFIED_ON,MODIFIED_BY) select {columnName},ISDELETED,MODIFIED_ON,MODIFIED_BY from tempView")
                df_history = spark.sql(f"describe history {destinationLakehouseName}.{destinationTableName} limit 1").first()
                if df_history:
                    rowsInserted = df_history["operationMetrics"]["numOutputRows"]
                    print(f"RowsInserted: {rowsInserted}, RowsUpdated: {rowsUpdated}")  
            else:
                print(f"No data inserted for {destinationLakehouseName}.{destinationTableName}")          


        endTime = getUTCDatetime()

        tempDF = createLogsEntry(sourceName, objectName, destinationLakehouseName, 
                 targetLayer, rowsInserted, rowsUpdated, masterPipelineRunID, startTime,endTime,currentAuditMaxId)
        return True, '', tempDF

    except Exception as e:
        print(f"Error during data load: {str(e)}")

StatementMeta(, f273984c-d2bf-4cc9-ad9b-2e54cb6233cb, 3, Finished, Available, Finished)

In [ ]:
def loadDataFromStageToBronze(sourceName,objectName,stageDF,inputLakehouseName,bronzeTableName,operationType,keyName,masterPipelineRunID,currentAuditMaxId):
    try:
        print("STAGE TO BRONZE loading started")
        isProcessed = 0
        targetLayer = 'bronze'
        bronzeTable = bronzeTableName.lower()
        bronzeFullName = f"{inputLakehouseName}.{bronzeTable}"
        tempViewName = f"{objectName}_stage_temp"
        stageDF.createOrReplaceTempView(tempViewName)
        bronzeDF = spark.table(bronzeFullName)
        bronzeColumns = [c.upper() for c in bronzeDF.columns] 

        if operationType.lower() == 'upsert':
            columnMapping = {}
            for c in bronzeColumns:
                if c == "ISDELETED":
                    columnMapping[c] = "CASE WHEN source.ISDELETED = 1 THEN 1 ELSE 0 END"
                elif c == "MODIFIED_ON":
                    columnMapping[c] = "source.ADDED_ON"
                elif c == "MODIFIED_BY":
                    columnMapping[c] = "source.ADDED_BY"
                else:
                    if c in [x.upper() for x in stageDF.columns]:
                        columnMapping[c] = f"source.{c}"
                    else:
                        # skip SYS_CHANGE_VERSION, SYS_CHANGE_OPERATION columns
                        pass
            keyList=keyName.split(',') 

            if len(keyList)>1:
                joinCondition = " AND ".join([f"target.{k}=source.{k}" for k in keyList])
            else:
                joinCondition = f"target.{keyName} = source.{keyName}"
            mergeColumnStatement = ", ".join([f"target.{k} = {v}" for k, v in columnMapping.items()])
            targetColumns = ", ".join(columnMapping.keys())

            srcColumns = ", ".join(columnMapping.values())
            merge_sql = f"""
                MERGE INTO {bronzeFullName} AS target
                USING {tempViewName} AS source
                ON {joinCondition}
                WHEN MATCHED THEN
                UPDATE SET {mergeColumnStatement}
                WHEN NOT MATCHED THEN
                INSERT ({targetColumns})
                VALUES ({srcColumns})
            """
            print("Generated MERGE SQL from Stage to Bronze:\n", merge_sql)
            
            # Execute merge
            spark.sql(merge_sql)

            print(f"Data from stage successfully merged into {bronzeFullName}")

            df_history = spark.sql(f"DESCRIBE HISTORY {bronzeFullName} LIMIT 1").first()
            if df_history:
                rowsInserted = df_history["operationMetrics"]["numTargetRowsInserted"]
                rowsUpdated = df_history["operationMetrics"]["numTargetRowsUpdated"]
                print(f"RowsInserted: {rowsInserted}, RowsUpdated: {rowsUpdated}")
            endTime = getUTCDatetime()

            tempDF = createLogsEntry(sourceName, objectName, inputLakehouseName, 
                    targetLayer, rowsInserted, rowsUpdated, masterPipelineRunID, startTime,endTime,currentAuditMaxId)
            return True, tempDF
        elif operationType.lower() == "overwrite":
            truncate_sql = f"DELETE FROM {bronzeFullName}"
            print("Truncating table:", truncate_sql)
            
            spark.sql(truncate_sql)

            insertColumns = []
            selectColumns = []
            for c in bronzeColumns:
                if c == "ISDELETED":
                    insertColumns.append(c)
                    selectColumns.append("CASE WHEN source.ISDELETED = 1 THEN 1 ELSE 0 END")
                elif c == "MODIFIED_ON":
                    insertColumns.append(c)
                    selectColumns.append("source.ADDED_ON")
                elif c == "MODIFIED_BY":
                    insertColumns.append(c)
                    selectColumns.append("source.ADDED_BY")
                else:
                    if c in [x.upper() for x in stageDF.columns]:
                        insertColumns.append(c)
                        selectColumns.append(f"source.{c}")
                    else:
                        pass

            insert_sql = f"""
                INSERT INTO {bronzeFullName} ({", ".join(insertColumns)})
                SELECT {", ".join(selectColumns)}
                FROM {tempViewName} AS source
            """

            print("Generated INSERT SQL for overwrite:\n", insert_sql)
            spark.sql(insert_sql)

            df_history = spark.sql(f"describe history {bronzeFullName} limit 1").first()
            if df_history:
                rowsUpdated = 0
                rowsInserted = df_history["operationMetrics"]["numOutputRows"]
                print(f"RowsInserted: {rowsInserted}, RowsUpdated: {rowsUpdated}")
                print(f"Data from stage successfully loaded into {bronzeFullName}")
            endTime = getUTCDatetime()

            tempDF = createLogsEntry(sourceName, objectName, inputLakehouseName, 
                    targetLayer, rowsInserted, rowsUpdated, masterPipelineRunID, startTime,endTime,currentAuditMaxId)
            return True, tempDF
    except Exception as e:
        print(f"Error during data load: {str(e)}")


    